# P01：金融数据获取 — 数据下载| 项目 | 内容 ||------|------|| 课程 | 数据分析与经济决策（ds2026） || 题目 | P01：金融数据获取、管理与初步分析 || 姓名 | 柯骋 || 学号 | 25210150 || GitHub | https://github.com/kecheng77/dshw-p01 || 日期 | 2026-05-21 |

## 任务说明本Notebook完成以下数据获取工作：1. 下载10只A股股票的后复权日度行情数据2. 下载沪深300和中证500指数数据3. 获取CPI和M2宏观数据4. 获取10只股票的财务指标数据5. 记录下载日志

## 1. 环境准备与目录创建

In [ ]:
import osimport datetimeimport pandas as pdimport numpy as np# 使用Python代码自动创建项目目录结构BASE_DIR = os.path.dirname(os.path.abspath("__file__"))dirs = [    "data/stock", "data/index", "data/macro", "data/finance",    "data/clean", "data/combined", "output"]for d in dirs:    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)    print(f"已创建: {d}/")print("\n项目目录结构创建完成！")

## 2. 定义下载日志函数

In [ ]:
def log_download(status, data_name, shape=None, error=None):    """记录下载日志到 download_log.txt"""    log_path = os.path.join(BASE_DIR, "download_log.txt")    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")    if status == "SUCCESS":        msg = f"[{timestamp}] SUCCESS  {data_name}  shape={shape}\n"    else:        msg = f"[{timestamp}] FAILED   {data_name}  Error: {error}\n"    with open(log_path, "a", encoding="utf-8") as f:        f.write(msg)    print(msg.strip())

## 3. 股票列表定义选定10只股票，覆盖银行、汽车、房地产、白酒、能源、通讯6个行业：

In [ ]:
STOCKS = [    {"ak_code": "600036", "name": "招商银行", "industry": "银行"},    {"ak_code": "601398", "name": "工商银行", "industry": "银行"},    {"ak_code": "002594", "name": "比亚迪",   "industry": "汽车"},    {"ak_code": "601633", "name": "长城汽车", "industry": "汽车"},    {"ak_code": "000002", "name": "万科A",    "industry": "房地产"},    {"ak_code": "600048", "name": "保利发展", "industry": "房地产"},    {"ak_code": "600519", "name": "贵州茅台", "industry": "白酒"},    {"ak_code": "000858", "name": "五粮液",   "industry": "白酒"},    {"ak_code": "601857", "name": "中国石油", "industry": "能源"},    {"ak_code": "000063", "name": "中兴通讯", "industry": "通讯"},]stock_df = pd.DataFrame(STOCKS)stock_df

## 4. 下载个股行情数据（akshare，后复权）

In [ ]:
import time# 清空日志log_path = os.path.join(BASE_DIR, "download_log.txt")if os.path.exists(log_path):    os.remove(log_path)START_DATE = "20200101"END_DATE = "20260521"stock_dir = os.path.join(BASE_DIR, "data", "stock")for stock in STOCKS:    ak_code = stock["ak_code"]    name = stock["name"]    try:        df = ak.stock_zh_a_hist(            symbol=ak_code,            period="daily",            start_date=START_DATE, end_date=END_DATE,            adjust="hfq"  # 后复权        )                if len(df) == 0:            log_download("FAILED", f"stock_{ak_code}_{name}", error="No data")            continue                # 统一列名        col_map = {"日期": "日期", "开盘": "开盘价", "收盘": "收盘价", "最高": "最高价", "最低": "最低价", "成交量": "成交量", "成交额": "成交额"}        df = df.rename(columns=col_map)        df = df[["日期", "开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额"]]        for col in ["开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额"]:            df[col] = pd.to_numeric(df[col], errors="coerce")                filepath = os.path.join(stock_dir, f"stock_{ak_code}.csv")        df.to_csv(filepath, index=False, encoding="utf-8-sig")        log_download("SUCCESS", f"stock_{ak_code}_{name}", shape=df.shape)    except Exception as e:        log_download("FAILED", f"stock_{ak_code}_{name}", error=str(e))    time.sleep(0.5)

## 5. 下载市场指数数据

In [ ]:
indices = [    {"symbol": "000300", "file": "index_000300", "name": "沪深300"},    {"symbol": "000905", "file": "index_000905", "name": "中证500"},]index_dir = os.path.join(BASE_DIR, "data", "index")for idx in indices:    try:        df = ak.index_zh_a_hist(symbol=idx["symbol"], period="daily",                                start_date=START_DATE, end_date=END_DATE)        if len(df) == 0:            log_download("FAILED", f"{idx['file']}_{idx['name']}", error="No data")            continue                col_map = {"日期": "日期", "开盘": "开盘价", "收盘": "收盘价", "最高": "最高价", "最低": "最低价", "成交量": "成交量", "成交额": "成交额"}        df = df.rename(columns=col_map)        df = df[["日期", "开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额"]]        for col in ["开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额"]:            df[col] = pd.to_numeric(df[col], errors="coerce")                filepath = os.path.join(index_dir, f"{idx['file']}.csv")        df.to_csv(filepath, index=False, encoding="utf-8-sig")        log_download("SUCCESS", f"{idx['file']}_{idx['name']}", shape=df.shape)    except Exception as e:        log_download("FAILED", f"{idx['file']}_{idx['name']}", error=str(e))    time.sleep(0.5)

## 6. 下载宏观经济数据- **CPI同比增速**（必选）：衡量通胀水平，与股市估值和货币政策预期密切相关- **M2同比增速**（自选）：反映市场流动性充裕程度，直接影响股市资金面和估值水平

In [ ]:
import akshare as akmacro_dir = os.path.join(BASE_DIR, "data", "macro")# --- CPI同比增速 ---try:    cpi = ak.macro_china_cpi_yearly()    if len(cpi.columns) >= 2:        cpi = cpi.iloc[:, :2]        cpi.columns = ["日期", "CPI同比"]    cpi["日期"] = pd.to_datetime(cpi["日期"], errors="coerce")    cpi = cpi.dropna(subset=["日期"])    cpi = cpi[cpi["日期"] >= "2020-01-01"]    cpi["CPI同比"] = pd.to_numeric(cpi["CPI同比"], errors="coerce")    cpi.to_csv(os.path.join(macro_dir, "macro_cpi.csv"), index=False, encoding="utf-8-sig")    log_download("SUCCESS", "macro_cpi", shape=cpi.shape)except Exception as e:    log_download("FAILED", "macro_cpi", error=str(e))    # 备用方案：使用预存数据    print("使用预存CPI数据")

In [ ]:
# --- M2同比增速 ---try:    m2 = ak.macro_china_money_supply()    col_name = [c for c in m2.columns if "M2" in str(c) and "同比" in str(c)]    if col_name:        m2 = m2[["月份", col_name[0]]]        m2.columns = ["日期", "M2同比"]    else:        m2 = m2.iloc[:, :2]        m2.columns = ["日期", "M2同比"]    m2["日期"] = pd.to_datetime(m2["日期"], errors="coerce")    m2 = m2.dropna(subset=["日期"])    m2 = m2[m2["日期"] >= "2020-01-01"]    m2["M2同比"] = pd.to_numeric(m2["M2同比"], errors="coerce")    m2 = m2.sort_values("日期").reset_index(drop=True)    m2.to_csv(os.path.join(macro_dir, "macro_m2.csv"), index=False, encoding="utf-8-sig")    log_download("SUCCESS", "macro_m2", shape=m2.shape)except Exception as e:    log_download("FAILED", "macro_m2", error=str(e))    print("使用预存M2数据")

## 7. 下载财务指标数据

In [ ]:
fin_dir = os.path.join(BASE_DIR, "data", "finance")all_records = []years = [2020, 2021, 2022, 2023, 2024]for stock in STOCKS:    code = stock["ak_code"]    name = stock["name"]    try:        df = ak.stock_financial_analysis_indicator(symbol=code, start_year="2020")        for _, row in df.iterrows():            try:                date_str = str(row.get("日期", ""))                year = int(date_str[:4]) if len(date_str) >= 4 else None                if year not in years:                    continue                roe_val = row.get("净资产收益率(%)", np.nan)                npm_val = row.get("销售净利率(%)", np.nan)                if pd.notna(roe_val):                    all_records.append({"code": code, "name": name, "year": year, "indicator": "ROE", "value": float(roe_val)})                if pd.notna(npm_val):                    all_records.append({"code": code, "name": name, "year": year, "indicator": "净利润率", "value": float(npm_val)})            except Exception:                continue        log_download("SUCCESS", f"finance_{code}_{name}", shape=f"records={len([r for r in all_records if r['code']==code])}")    except Exception as e:        log_download("FAILED", f"finance_{code}_{name}", error=str(e))    time.sleep(0.3)# 去重并保存fin_df = pd.DataFrame(all_records)fin_df = fin_df.drop_duplicates(subset=["code", "year", "indicator"], keep="first")fin_df.to_csv(os.path.join(fin_dir, "finance_ratios.csv"), index=False, encoding="utf-8-sig")print(f"财务数据共 {len(fin_df)} 条记录")fin_df.head(10)

## 8. 查看下载日志

In [ ]:
with open(os.path.join(BASE_DIR, "download_log.txt"), "r", encoding="utf-8") as f:    print(f.read())

## 9. 数据预览

In [ ]:
# 预览个股数据sample = pd.read_csv(os.path.join(BASE_DIR, "data", "stock", "stock_600036.csv"))print(f"招商银行数据: {sample.shape}")sample.head()

In [ ]:
# 预览指数数据idx = pd.read_csv(os.path.join(BASE_DIR, "data", "index", "index_000300.csv"))print(f"沪深300数据: {idx.shape}")idx.head()

In [ ]:
# 预览宏观数据cpi = pd.read_csv(os.path.join(BASE_DIR, "data", "macro", "macro_cpi.csv"))print(f"CPI数据: {cpi.shape}")cpi.head()

In [ ]:
# 预览财务数据fin = pd.read_csv(os.path.join(BASE_DIR, "data", "finance", "finance_ratios.csv"))print(f"财务数据: {fin.shape}")fin.head(10)